walkthrough for docs

In [1]:
from pathlib import Path
from metasmith.python_api import *
from metasmith import examples

from local.constants import WORKSPACE_ROOT

In [2]:
EXAMPLES_DIR = WORKSPACE_ROOT/"src/metasmith/example_resources"
dtypes = DataTypeLibrary.Load(EXAMPLES_DIR/"types/minimal_genomics.yml")

xgdb_path = EXAMPLES_DIR/"data/fosmid.xgdb"
xgdb = DataInstanceLibrary.Load(xgdb_path)
# xgdb = DataInstanceLibrary(xgdb_path)
# xgdb.AddTypeLibrary("genomics", dtypes)
# xgdb.Add([
#     (EXAMPLES_DIR/"fosmid.fna", "./fosmid.fna", "genomics::contigs"),
# ])
# xgdb.PruneTypes()
# xgdb.Save()

refdb_path = EXAMPLES_DIR/"data/references.xgdb"
refdb = DataInstanceLibrary.Load(refdb_path)
# refdb = DataInstanceLibrary(refdb_path)
# refdb.AddTypeLibrary("genomics", dtypes)
# refdb.Add([
#     (EXAMPLES_DIR/"swissprot_bcaa.fna", "./swissprot_bcaa.fna", "genomics::aa_sequences"),
# ])
# refdb.PruneTypes()
# refdb.Save()

trans_path = EXAMPLES_DIR/"transforms/gene_annotation"
transforms = TransformInstanceLibrary.Load(trans_path); # transforms.PruneTypes()

# transforms = TransformInstanceLibrary(trans_path)
# transforms.AddTypeLibrary("genomics", dtypes)
# transforms.AddStub("pprodigal")
# transforms.AddStub("diamond")
# transforms.AddStub("make_diamond_db")
# transforms.Save()

In [3]:
agent = Agent(
    home = Source.FromLocal(Path("./cache/local_home").resolve()),
    # home = Source.FromLocal((WORKSPACE_ROOT/"docs/source/metasmith_home").resolve()),
)
agent.Deploy()

2025-06-17_18-09-53  | >>> AGENT_HOME=/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home
2025-06-17_18-09-53  | >>> mkdir -p $AGENT_HOME
2025-06-17_18-09-53  | >>> mkdir -p /home/tony/.globus
2025-06-17_18-09-53  | >>> mkdir -p /home/tony/.globusonline
2025-06-17_18-09-53  | >>> {if not exists}: apptainer pull/metasmith.sif docker://quay.io/hallamlab/metasmith:latest
2025-06-17_18-09-53  | staged [msm_stub]
2025-06-17_18-09-53  | staged [msm]
2025-06-17_18-09-53  | staged [lib/agent.yml]
2025-06-17_18-09-53  | staged [lib/msm_bootstrap]
2025-06-17_18-09-53  | staged [lib/nextflow_config]
2025-06-17_18-09-53  | deploying [5] staged files
2025-06-17_18-09-53  | >>> cd /home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home && ./msm api deploy_from_container
2025-06-17_18-09-53  | including dev binds
2025-06-17_18-09-54  | 2025-06-17_18-09-54  | api call to [deploy_from_container] with [{}]
2025-06-17_18-09-54  | 2025-06-17_18-09-54  | deploying to [/ws]
2

In [4]:
task = agent.GenerateWorkflow(
    given=[xgdb, refdb],
    transforms=[transforms],
    targets=[
        dtypes["orf_annotations"].WithLineage([dtypes["contigs"]]),
    ]
)
for step in task.plan.steps:
    print(f">>> {step.transform.name}")
    for x in step.uses:
        print(x.path)
    print("---")
    for x in step.produces:
        print(x.path)
    print()

>>> prodigal
fosmid.fna
prodigal.oci.uri
---
orfs.faa

>>> blast
orfs.faa
swissprot_bcaa.faa
blast.oci.uri
---
annotations.csv



In [5]:
agent.StageWorkflow(task, on_exist="clear")
# agent.StageWorkflow(task, on_exist="update")
# agent.StageWorkflow(task)

2025-06-17_18-09-55  | connecting to deployed agent
2025-06-17_18-09-55  | starting relay service


E| > 2025-06-17_18-09-59 E| Failed to connect to server
E| > 2025-06-17_18-09-59 E| removing stale connections at [relay/connections]


 | > 2025-06-17_18-09-59  | relay server started with pid: [216675]
2025-06-17_18-09-59 W| task already staged at [/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/runs/O7u9btAO]
2025-06-17_18-09-59 W| clearing previously staged task
2025-06-17_18-09-59  | sending metadata for workflow [O7u9btAO]
2025-06-17_18-10-02  | staging
 | > including dev binds
 | > 2025-06-17_18-10-03  | api call to [stage_workflow] with [{'task_key': 'O7u9btAO'}]
 | > 2025-06-17_18-10-03  | staging workflow [O7u9btAO] with [2] data libs and [1] transform libs
 | > 2025-06-17_18-10-03  | ex| /home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home
 | > 2025-06-17_18-10-03  | work [/ws/runs/O7u9btAO]
 | > 2025-06-17_18-10-03  | data [/msm_home/data]
 | > 2025-06-17_18-10-03  | external work [/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/runs/O7u9btAO]
 | > 2025-06-17_18-10-03  | external data [/home/tony/workspace/tools/Metasmith/main/local_mock/cache/loc

In [6]:
agent.RunWorkflow(task)

2025-06-17_18-10-06  | connecting to deployed agent
2025-06-17_18-10-06  | starting relay service
 | > 2025-06-17_18-10-07  | connecting to relay as [tUwiKBzm97Jh]


E| > 2025-06-17_18-10-07 E| relay server already running in [relay/connections]


2025-06-17_18-10-07  | executing workflow [O7u9btAO]
2025-06-17_18-10-07  | closing connection


In [7]:
agent.CheckWorkflow(task)

2025-06-17_18-10-44  | connecting to deployed agent
2025-06-17_18-10-45  | starting relay service
 | > 2025-06-17_18-10-45  | connecting to relay as [WUYUBahczVnq]


E| > 2025-06-17_18-10-45 E| relay server already running in [relay/connections]


 | > including dev binds
 | > 2025-06-17_18-10-46  | api call to [check_workflow] with [{'key': 'O7u9btAO'}]
 | > 2025-06-17_18-10-46  | searching for logs
 | > 2025-06-17_18-10-46  | found [1] runs
 | > 2025-06-17_18-10-46  |     1: [logs.2025-06-17_18-10-07]
 | > 2025-06-17_18-10-46  | here is the main log of the latest run [logs.2025-06-17_18-10-07]
 | > 2025-06-17_18-10-46  | >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
 | > 2025-06-17_18-10-46  | 
 | > including dev binds
 | > 2025-06-17_18-10-08  | api call to [run_workflow] with [{'key': 'O7u9btAO', 'log_dir': '_metasmith/logs.2025-06-17_18-10-07'}]
 | > 2025-06-17_18-10-08  | start time [2025-06-17_18-10-08]
 | > 2025-06-17_18-10-08  | running workflow [O7u9btAO] with preset [default]
 | > 2025-06-17_18-10-08  | loading agent metadata
 | > 2025-06-17_18-10-08  | workspace [/msm_home/runs/O7u9btAO]
 | > 2025-06-17_18-10-08  | external workspace [/home/tony/workspace/tools/Metasmith/main/local_mock/cache/loca